# Task 8: Multi-Agent Swarm with Shared Transactional Blackboard Architecture

## Overview
Decentralized specialist agents coordinating via a shared memory store with state locks.


In [ ]:
# Redis-backed transactional blackboard simulation and specialist agent runner
class RedisBlackboardSimulation:
    def __init__(self):
        self.state = {}
        self.locks = set()

    def acquire_lock(self, key: str, agent_id: str):
        if key in self.locks:
            return False
        self.locks.add(key)
        return True

    def release_lock(self, key: str, agent_id: str):
        self.locks.discard(key)

    def write(self, key: str, value: str, agent_id: str):
        if key in self.locks:
            self.state[key] = value
            return True
        return False

class SpecialistAgent:
    def __init__(self, agent_id, role, blackboard):
        self.agent_id = agent_id
        self.role = role
        self.blackboard = blackboard

    def execute_task(self, key, data):
        if self.blackboard.acquire_lock(key, self.agent_id):
            self.blackboard.write(key, f"[{self.role}] {data}", self.agent_id)
            self.blackboard.release_lock(key, self.agent_id)
            return True
        return False


In [ ]:
# Test multi-agent coordination over shared blackboard memory
bb = RedisBlackboardSimulation()
agent1 = SpecialistAgent("A1", "CodeGenerator", bb)
agent2 = SpecialistAgent("A2", "SystemAuditor", bb)

agent1.execute_task("state_key", "Generated code")
agent2.execute_task("state_key", "Audited code")

print("Blackboard State:", bb.state)
print("[SUCCESS] Multi-Agent Blackboard state updates verified.")
